# Modelagem Sísmica utilizando equação da onda acústica de primeira ordem

In [31]:
import numpy as np
import sympy as sp

In [32]:
import matplotlib.pyplot as plt

from devito import *
from examples.seismic.source import RickerSource, WaveletSource, TimeAxis
from examples.seismic import ModelViscoacoustic, plot_image, setup_geometry, plot_velocity

In [33]:
nx = 1000
nz = 200

shape = (nx, nz)
spacing = (20., 20.)
origin = (0., 0.)
nlayers = 3
nbl = 1000
space_order = 8
dtype = np.float32

v = np.zeros(shape)
rho = np.zeros(shape)
qp = np.zeros(shape)


v[:, :71] = 1.5
v[:, 71:101] = 2.0
v[:, 101:] = 4.5

# vp_top = 1.5
# vp_bottom = 3.5

# # Define a velocity profile in km/s
# v = np.empty(shape, dtype=dtype)
# v[:] = vp_top  # Top velocity (background)
# vp_i = np.linspace(vp_top, vp_bottom, nlayers)
# for i in range(1, nlayers):
#     v[..., i*int(shape[-1] / nlayers):] = vp_i[i]  # Bottom velocity

rho[:] = 1. #0.31*(v[:]*1000.)**0.25 # Gardner's relation
qp[:] = 1000.

In [ ]:
model = ModelViscoacoustic(space_order=space_order, vp=v, qp=qp, b=1/rho, 
                           origin=origin, shape=shape, spacing=spacing, 
                           nbl=nbl)

In [ ]:
#NBVAL_IGNORE_OUTPUT
aspect_ratio = model.shape[0]/model.shape[1]

plt_options_model = {'cmap': 'jet', 'extent': [model.origin[0], model.origin[0] + model.domain_size[0],
                                               model.origin[1] + model.domain_size[1], model.origin[1]]}
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))

slices = [slice(model.nbl, -model.nbl), slice(model.nbl, -model.nbl)]

img1 = ax[0].imshow(np.transpose(model.vp.data[slices]), vmin=1.5, vmax=3.5, **plt_options_model)
fig.colorbar(img1, ax=ax[0])
ax[0].set_title(r"V (km/s)", fontsize=20)
ax[0].set_xlabel('X (m)', fontsize=20)
ax[0].set_ylabel('Depth (m)', fontsize=20)
ax[0].set_aspect('auto')

img2 = ax[1].imshow(np.transpose(qp), vmin=15, vmax=220, **plt_options_model)
fig.colorbar(img2, ax=ax[1])
ax[1].set_title("Q", fontsize=20)
ax[1].set_xlabel('X (m)', fontsize=20)
ax[1].set_ylabel('Depth (m)', fontsize=20)
ax[1].set_aspect('auto')

img3 = ax[2].imshow(np.transpose(rho), vmin=1.9, vmax=2.4, **plt_options_model)
fig.colorbar(img3, ax=ax[2])
ax[2].set_title(r"Density $\rho$ (g/cm^3)", fontsize=20)
ax[2].set_xlabel('X (m)', fontsize=20)
ax[2].set_ylabel('Depth (m)', fontsize=20)
ax[2].set_aspect('auto')

plt.tight_layout()

In [36]:
f0 = 0.010 # peak/dominant frequency 
b = model.b
rho = 1./b

# velocity model
vp = model.vp

s = model.grid.stepping_dim.spacing
damp = model.damp

In [37]:
# Time step in ms and time range:
t0, tn = 0., 6000.
dt = model.critical_dt
time_range = TimeAxis(start=t0, stop=tn, step=dt)

In [38]:
from examples.seismic import Receiver

In [39]:
src = RickerSource(name='src', grid=model.grid, f0=f0, time_range=time_range)
src.coordinates.data[0, :] = 10.
src.coordinates.data[0, -1] = 0.  

# Create symbol for receivers
rec = Receiver(name='rec', grid=model.grid, npoint=shape[0], time_range=time_range)

# Prescribe even spacing for receivers along the x-axis
rec.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=shape[0])
rec.coordinates.data[:, 1] = 0.

In [40]:
P = TimeFunction(name="P", grid=model.grid, time_order=1, space_order=space_order, 
                     staggered=NODE)

v = VectorTimeFunction(name="v", grid=model.grid, time_order=1, space_order=space_order)

In [ ]:
P

In [ ]:
v

In [ ]:
EDP_P = P.dt + rho * (vp**2) * div(v.forward)
st_P = Eq(P.forward, solve(EDP_P, P.forward))

display(EDP_P, st_P)

In [ ]:
EDP_v  = v.dt + b * grad(P)
st_v = Eq(v.forward, solve(EDP_v, v.forward))

display(EDP_v, st_v)

In [45]:
src_term = src.inject(field=P.forward, expr=(s*src))

In [46]:
rec_term = rec.interpolate(expr=P.forward)

In [ ]:
op = Operator([st_v, st_P] + src_term + rec_term, subs=model.spacing_map)

op(time=time_range.num-1, dt=dt, src=src, rec=rec)

In [ ]:
from examples.seismic import plot_shotrecord

plot_shotrecord(rec.data, model, t0, tn)

In [ ]:
from examples.seismic.curso_capacitacao.utils import plot_shotrecord_utils

scale = 0.002 * np.max(np.abs(rec.data[:]))

plot_shotrecord_utils(rec.data, model, 0, tn, scale)